# Exercise 3: Hosting a Graph Database on AWS

In this exercise, you will learn how to launch and interact with graph databases on AWS. We'll explore two approaches:

1. **AWS Neptune** — Amazon's fully managed graph database service with built-in security, continuous backups, and AWS integrations
2. **Neo4j on AWS** — Self-hosted Neo4j instance running on an EC2 instance, leveraging Neo4j's AWS cloud partnership

Just like RDBMS (RDS) and NoSQL (DynamoDB/DocumentDB) systems, graph databases face the same hosting challenges:
- **Networking**: VPCs, subnets, security groups
- **Security**: Authentication, encryption, access control
- **Availability**: Backups, failover, replication
- **Scaling**: Read replicas, instance sizing

## Prerequisites

- AWS Account with appropriate permissions (Neptune, EC2, VPC, IAM)
- Python 3.9+ with boto3 installed
- AWS credentials configured

## Cost Warning ⚠️

**IMPORTANT:** This exercise creates AWS resources that may incur charges. Neptune instances start at ~$0.10/hr for the smallest instance. Make sure to run the **Clean Up** section at the end to delete all resources!

## Step 1: AWS Authentication and Setup

First, we'll configure our AWS credentials and create the service clients we need.

In [ ]:
%%capture
%pip install boto3 pandas

import boto3
import json
import time
import pandas as pd
from botocore.exceptions import ClientError

### Configure AWS Credentials

#### Task 1: Enter your AWS credentials below

Copy your AWS credentials from the AWS Academy Learner Lab or your AWS account.

In [ ]:
import os

# Set your AWS credentials
### BEGIN SOLUTION
os.environ['AWS_ACCESS_KEY_ID'] = 'YOUR_ACCESS_KEY_HERE'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'YOUR_SECRET_KEY_HERE'
os.environ['AWS_SESSION_TOKEN'] = 'YOUR_SESSION_TOKEN_HERE'
### END SOLUTION

REGION = 'us-east-1'

# Create service clients
session = boto3.Session(region_name=REGION)
neptune_client = session.client('neptune')
ec2_client = session.client('ec2')
iam_client = session.client('iam')
sts_client = session.client('sts')

# Verify connection
try:
    identity = sts_client.get_caller_identity()
    print(f"✅ Authenticated as: {identity['Arn']}")
    print(f"   Account: {identity['Account']}")
    print(f"   Region: {REGION}")
except Exception as e:
    print(f"❌ Authentication failed: {e}")

## Step 2: Network Infrastructure Setup

Graph databases on AWS need the same networking foundations as any managed database:
- A **VPC** to isolate resources
- **Subnets** across availability zones (required by Neptune)
- **Security Groups** to control access
- A **DB Subnet Group** to tell Neptune which subnets to use

This is identical to what we did for RDS and DocumentDB — the infrastructure patterns are the same regardless of database type.

In [ ]:
# Discover default VPC and subnets
vpcs = ec2_client.describe_vpcs(Filters=[{'Name': 'isDefault', 'Values': ['true']}])
vpc_id = vpcs['Vpcs'][0]['VpcId']
print(f"✅ Default VPC: {vpc_id}")

# Get subnets in at least 2 availability zones (required by Neptune)
subnets = ec2_client.describe_subnets(
    Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]
)['Subnets']

# Neptune requires subnets in at least 2 AZs
az_subnets = {}
for subnet in subnets:
    az = subnet['AvailabilityZone']
    if az not in az_subnets:
        az_subnets[az] = subnet['SubnetId']

subnet_ids = list(az_subnets.values())[:3]
print(f"✅ Subnets across {len(subnet_ids)} AZs: {subnet_ids}")

In [ ]:
# Create Security Group for Neptune (port 8182)
SG_NAME = 'neptune-exercise-sg'

try:
    sg_response = ec2_client.create_security_group(
        GroupName=SG_NAME,
        Description='Security group for Neptune and Neo4j graph databases',
        VpcId=vpc_id
    )
    sg_id = sg_response['GroupId']
    print(f"✅ Created security group: {sg_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'InvalidGroup.Duplicate':
        sgs = ec2_client.describe_security_groups(
            Filters=[{'Name': 'group-name', 'Values': [SG_NAME]}]
        )['SecurityGroups']
        sg_id = sgs[0]['GroupId']
        print(f"⚠️ Security group already exists: {sg_id}")
    else:
        raise

# Allow inbound traffic on Neptune (8182) and Neo4j (7474, 7687) ports
ports = [
    (8182, 'Neptune HTTPS/WebSocket'),
    (7474, 'Neo4j HTTP'),
    (7687, 'Neo4j Bolt'),
]

for port, description in ports:
    try:
        ec2_client.authorize_security_group_ingress(
            GroupId=sg_id,
            IpPermissions=[{
                'IpProtocol': 'tcp',
                'FromPort': port,
                'ToPort': port,
                'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': description}]
            }]
        )
        print(f"   ✅ Allowed port {port} ({description})")
    except ClientError as e:
        if 'InvalidPermission.Duplicate' in str(e):
            print(f"   ⚠️ Port {port} rule already exists")
        else:
            raise

In [ ]:
# Create DB Subnet Group for Neptune
SUBNET_GROUP_NAME = 'neptune-exercise-subnet-group'

try:
    neptune_client.create_db_subnet_group(
        DBSubnetGroupName=SUBNET_GROUP_NAME,
        DBSubnetGroupDescription='Subnet group for Neptune graph database exercise',
        SubnetIds=subnet_ids
    )
    print(f"✅ Created DB subnet group: {SUBNET_GROUP_NAME}")
except ClientError as e:
    if 'DBSubnetGroupAlreadyExists' in str(e):
        print(f"⚠️ Subnet group already exists: {SUBNET_GROUP_NAME}")
    else:
        raise

## Part A: AWS Neptune — Amazon's Managed Graph Database

### What is AWS Neptune?

- **Fully managed** graph database service — no server patching, backups, or replication to handle
- **Built-in security** — encryption at rest and in transit, VPC isolation, IAM authentication
- **Continuous backups** — automatic backups to S3 with point-in-time recovery
- **Supports multiple query languages**: Apache TinkerPop Gremlin (property graphs) and SPARQL (RDF graphs)
- **High availability** — read replicas across AZs, automatic failover
- Best for: Applications needing a managed, highly available graph database integrated with AWS services

### Neptune vs Neo4j

| Feature | AWS Neptune | Neo4j |
|-|-|-|
| Query Language | Gremlin / SPARQL / openCypher | Cypher |
| Hosting | AWS managed only | Self-hosted, cloud, or Aura (managed) |
| Management | Fully managed | You manage (or use Aura) |
| AWS Integration | Native (IAM, CloudWatch, S3) | Manual |
| Scaling | Read replicas | Causal clustering |
| Cost | Pay per instance-hour | Community (free) or Enterprise license |

### Step 3: Create a Neptune Cluster

Neptune uses a cluster architecture (similar to Aurora/DocumentDB):
- A **Cluster** manages storage and replication
- An **Instance** within the cluster handles compute (reads/writes)

#### Task 2: Choose a name for your Neptune cluster

In [ ]:
### BEGIN SOLUTION
NEPTUNE_CLUSTER_ID = 'graph-exercise-cluster'
### END SOLUTION

# Create Neptune cluster
try:
    response = neptune_client.create_db_cluster(
        DBClusterIdentifier=NEPTUNE_CLUSTER_ID,
        Engine='neptune',
        DBSubnetGroupName=SUBNET_GROUP_NAME,
        VpcSecurityGroupIds=[sg_id],
        StorageEncrypted=True,
        DeletionProtection=False,
        Tags=[{'Key': 'Project', 'Value': 'data-modeling-exercise'}]
    )
    print(f"✅ Neptune cluster creation initiated: {NEPTUNE_CLUSTER_ID}")
    print(f"   Status: {response['DBCluster']['Status']}")
    print(f"   Engine: {response['DBCluster']['Engine']}")
except ClientError as e:
    if 'DBClusterAlreadyExistsFault' in str(e):
        print(f"⚠️ Cluster already exists: {NEPTUNE_CLUSTER_ID}")
    else:
        raise

### Create a Neptune Instance

Now let's add a compute instance to our cluster. We'll use the smallest available instance type (`db.t3.medium`) to minimize cost.

In [ ]:
NEPTUNE_INSTANCE_ID = 'graph-exercise-instance-1'

try:
    response = neptune_client.create_db_instance(
        DBInstanceIdentifier=NEPTUNE_INSTANCE_ID,
        DBClusterIdentifier=NEPTUNE_CLUSTER_ID,
        DBInstanceClass='db.t3.medium',
        Engine='neptune',
        Tags=[{'Key': 'Project', 'Value': 'data-modeling-exercise'}]
    )
    print(f"✅ Neptune instance creation initiated: {NEPTUNE_INSTANCE_ID}")
    print(f"   Instance class: db.t3.medium")
    print(f"   Status: {response['DBInstance']['DBInstanceStatus']}")
except ClientError as e:
    if 'DBInstanceAlreadyExists' in str(e):
        print(f"⚠️ Instance already exists: {NEPTUNE_INSTANCE_ID}")
    else:
        raise

### Wait for Neptune to Become Available

Neptune instances typically take **10-15 minutes** to become available. This is similar to RDS and DocumentDB — managed database services need time to provision compute, configure networking, and initialize storage.

⏳ **Be patient** — this is a good time to read ahead about the Gremlin query language!

In [ ]:
def wait_for_neptune_available(client, instance_id, max_wait=1200):
    """Wait for Neptune instance to become available."""
    start_time = time.time()
    while time.time() - start_time < max_wait:
        try:
            response = client.describe_db_instances(
                DBInstanceIdentifier=instance_id
            )
            status = response['DBInstances'][0]['DBInstanceStatus']
            elapsed = int(time.time() - start_time)
            print(f"   ⏳ Status: {status} ({elapsed}s elapsed)")

            if status == 'available':
                endpoint = response['DBInstances'][0]['Endpoint']['Address']
                port = response['DBInstances'][0]['Endpoint']['Port']
                return endpoint, port
        except Exception as e:
            print(f"   Waiting... ({e})")

        time.sleep(30)

    raise TimeoutError(f"Neptune instance did not become available within {max_wait}s")

print("⏳ Waiting for Neptune instance to become available...")
print("   (This typically takes 10-15 minutes)")
neptune_endpoint, neptune_port = wait_for_neptune_available(neptune_client, NEPTUNE_INSTANCE_ID)
print(f"\n✅ Neptune is available!")
print(f"   Endpoint: {neptune_endpoint}")
print(f"   Port: {neptune_port}")

### Step 4: Query Neptune with Gremlin

Neptune supports **Gremlin** (Apache TinkerPop) for property graph queries. Gremlin uses a traversal-based syntax:
- `g.addV('label')` — Add a vertex (node)
- `g.V()` — Get all vertices
- `g.addE('label')` — Add an edge (relationship)
- `.property('key', 'value')` — Set properties
- `.has('key', 'value')` — Filter by property

Neptune also supports **openCypher** — the open-source version of Neo4j's query language!

#### Task 3: Interact with Neptune using the Gremlin HTTP API

We'll use Neptune's HTTP REST endpoint to send Gremlin queries.

In [ ]:
import urllib.request
import urllib.parse

def gremlin_query(endpoint, port, query):
    """Send a Gremlin query to Neptune via HTTP."""
    url = f"https://{endpoint}:{port}/gremlin"
    data = json.dumps({'gremlin': query}).encode('utf-8')
    req = urllib.request.Request(
        url,
        data=data,
        headers={'Content-Type': 'application/json'}
    )
    try:
        with urllib.request.urlopen(req) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"❌ Query error: {e}")
        return None

# Note: Neptune is VPC-only by default. To query from outside the VPC,
# you would need a bastion host, VPN, or Neptune Notebook.
# For this exercise, we'll demonstrate the query patterns conceptually.

print("Neptune Gremlin Query Examples:")
print("="*50)

# Example Gremlin queries (these would run if connected within the VPC)
gremlin_examples = [
    {
        'description': 'Add a Customer vertex',
        'query': "g.addV('Customer').property('name', 'Alice').property('status', 'gold')"
    },
    {
        'description': 'Add a Product vertex',
        'query': "g.addV('Product').property('name', 'Small Gear').property('price', 49.99)"
    },
    {
        'description': 'Create a PURCHASED edge',
        'query': "g.V().has('Customer','name','Alice').addE('PURCHASED').to(g.V().has('Product','name','Small Gear')).property('quantity', 5)"
    },
    {
        'description': 'Find all customers who purchased a product',
        'query': "g.V().hasLabel('Product').has('name','Small Gear').in('PURCHASED').values('name')"
    },
    {
        'description': 'Count all vertices by label',
        'query': "g.V().groupCount().by(label)"
    }
]

for ex in gremlin_examples:
    print(f"\n📌 {ex['description']}:")
    print(f"   {ex['query']}")

### Neptune openCypher Support

Neptune also supports **openCypher** queries — the same Cypher syntax you used with Neo4j! This means your existing Cypher knowledge transfers directly.

#### Task 4: Write an openCypher query for Neptune

The same MATCH patterns from Neo4j work in Neptune's openCypher endpoint.

In [ ]:
# openCypher queries work on Neptune's /openCypher endpoint
# These are the same Cypher queries from our Neo4j exercises!

### BEGIN SOLUTION
opencypher_examples = [
    {
        'description': 'Create a Customer node',
        'query': "CREATE (c:Customer {name: 'Alice', status: 'gold', email: 'alice@company.com'})"
    },
    {
        'description': 'Find all gold customers',
        'query': "MATCH (c:Customer {status: 'gold'}) RETURN c.name, c.email"
    },
    {
        'description': 'Create a purchase relationship',
        'query': "MATCH (c:Customer {name: 'Alice'}), (p:Product {name: 'Small Gear'}) CREATE (c)-[:PURCHASED {qty: 5}]->(p)"
    },
    {
        'description': 'Find products purchased by a customer',
        'query': "MATCH (c:Customer {name: 'Alice'})-[:PURCHASED]->(p:Product) RETURN p.name, p.price"
    }
]
### END SOLUTION

print("Neptune openCypher Query Examples:")
print("="*50)
for ex in opencypher_examples:
    print(f"\n📌 {ex['description']}:")
    print(f"   {ex['query']}")

print("\n💡 Notice: These are identical to the Cypher queries from Exercises 1 and 2!")
print("   Your Neo4j skills transfer directly to Neptune.")

### Check Neptune Cluster Status

Let's verify our Neptune cluster details and explore its configuration.

In [ ]:
# Check Neptune cluster status
try:
    cluster_info = neptune_client.describe_db_clusters(
        DBClusterIdentifier=NEPTUNE_CLUSTER_ID
    )['DBClusters'][0]

    print("Neptune Cluster Details:")
    print("=" * 50)
    print(f"  Cluster ID:    {cluster_info['DBClusterIdentifier']}")
    print(f"  Status:        {cluster_info['Status']}")
    print(f"  Engine:        {cluster_info['Engine']}")
    print(f"  Engine Ver:    {cluster_info['EngineVersion']}")
    print(f"  Endpoint:      {cluster_info['Endpoint']}")
    print(f"  Port:          {cluster_info['Port']}")
    print(f"  Encrypted:     {cluster_info['StorageEncrypted']}")
    print(f"  Multi-AZ:      {cluster_info.get('MultiAZ', 'N/A')}")
    print(f"  Backup Ret.:   {cluster_info['BackupRetentionPeriod']} days")

    # Instance details
    instance_info = neptune_client.describe_db_instances(
        DBInstanceIdentifier=NEPTUNE_INSTANCE_ID
    )['DBInstances'][0]

    print(f"\nNeptune Instance Details:")
    print(f"  Instance ID:   {instance_info['DBInstanceIdentifier']}")
    print(f"  Instance Class:{instance_info['DBInstanceClass']}")
    print(f"  Status:        {instance_info['DBInstanceStatus']}")
    print(f"  AZ:            {instance_info['AvailabilityZone']}")

except ClientError as e:
    print(f"❌ Could not describe cluster: {e}")

## Part B: Neo4j on AWS — Self-Hosted via AWS Marketplace

### Why Host Neo4j on AWS?

Neo4j is an **AWS Partner** and offers:
- **AMI on AWS Marketplace** — Pre-configured EC2 images
- **Neo4j Aura on AWS** — Neo4j's own managed service running on AWS infrastructure
- **CloudFormation templates** — Infrastructure-as-code deployment

### Self-Hosting Challenges

When you self-host Neo4j (vs using Neptune), **you** are responsible for:
- Installing and updating Neo4j
- Configuring backups and recovery
- Managing security patches
- Setting up monitoring and alerts
- Handling scaling and replication

These are the **same challenges** as self-hosting PostgreSQL vs using RDS, or self-hosting MongoDB vs using DocumentDB.

### Managed vs Self-Hosted Comparison

| Responsibility | Neptune (Managed) | Neo4j on EC2 (Self-Hosted) | Neo4j Aura (Managed) |
|-|-|-|-|
| Installation | AWS handles | You handle | Neo4j handles |
| Patching/Updates | Automatic | Manual | Automatic |
| Backups | Automated to S3 | You configure | Automated |
| Monitoring | CloudWatch built-in | You set up | Built-in dashboard |
| Scaling | Add read replicas | Manual resize | Auto-scaling |
| Query Language | Gremlin/SPARQL/openCypher | Cypher | Cypher |
| Cost | Per instance-hour | EC2 + storage | Subscription |

### Step 5: Discover Neo4j AMIs on AWS Marketplace

Neo4j publishes official AMIs (Amazon Machine Images) in the AWS Marketplace. Let's explore what's available.

#### Task 5: Search for Neo4j AMIs

In [ ]:
### BEGIN SOLUTION
# Search for Neo4j AMIs in the marketplace
try:
    neo4j_images = ec2_client.describe_images(
        Filters=[
            {'Name': 'name', 'Values': ['*neo4j*']},
            {'Name': 'state', 'Values': ['available']}
        ],
        Owners=['aws-marketplace']
    )['Images']

    if neo4j_images:
        print(f"✅ Found {len(neo4j_images)} Neo4j AMIs on AWS Marketplace")
        print("\nTop 5 most recent:")
        sorted_images = sorted(neo4j_images, key=lambda x: x.get('CreationDate', ''), reverse=True)[:5]
        for img in sorted_images:
            print(f"  📦 {img['Name'][:80]}")
            print(f"     AMI ID: {img['ImageId']}")
            print(f"     Created: {img.get('CreationDate', 'N/A')}")
            print()
    else:
        print("⚠️ No Neo4j AMIs found. They may be available in the AWS Marketplace console.")
        print("   Visit: https://aws.amazon.com/marketplace/search/?searchTerms=neo4j")

except ClientError as e:
    print(f"❌ Error searching AMIs: {e}")
### END SOLUTION

### Neo4j Deployment Options on AWS

There are several ways to run Neo4j on AWS:

#### Option 1: EC2 with Neo4j AMI
```bash
# Launch an EC2 instance with Neo4j AMI from Marketplace
aws ec2 run-instances \
    --image-id ami-xxxxx \
    --instance-type t3.medium \
    --key-name your-key-pair \
    --security-group-ids sg-xxxxx
```

#### Option 2: Neo4j Aura (Managed)
```
# No infrastructure to manage!
# Sign up at: https://neo4j.com/cloud/aura/
# Choose AWS as the cloud provider
# Get a connection URI like: neo4j+s://xxxxx.databases.neo4j.io
```

#### Option 3: Docker on EC2/ECS
```bash
# Run Neo4j in a Docker container on EC2
docker run -d \
    --name neo4j \
    -p 7474:7474 -p 7687:7687 \
    -e NEO4J_AUTH=neo4j/password \
    neo4j:latest
```

## Step 6: Comparing Graph Database Hosting — Key Considerations

Let's create a comprehensive comparison of the hosting options we've explored.

In [ ]:
# Comprehensive hosting comparison
comparison_data = {
    'Feature': [
        'Type',
        'Query Languages',
        'Management',
        'Setup Time',
        'Backups',
        'Security',
        'Scaling',
        'Monitoring',
        'AWS Integration',
        'Cost Model',
        'Best For'
    ],
    'AWS Neptune': [
        'Managed Service',
        'Gremlin, SPARQL, openCypher',
        'Fully managed by AWS',
        '10-15 minutes',
        'Automatic, continuous to S3',
        'IAM, VPC, encryption at rest/transit',
        'Read replicas (up to 15)',
        'CloudWatch metrics built-in',
        'Native (IAM, S3, Lambda, etc.)',
        'Pay per instance-hour',
        'AWS-native apps, multi-query-language'
    ],
    'Neo4j on EC2': [
        'Self-Hosted',
        'Cypher',
        'You manage everything',
        '30-60 minutes',
        'You configure (EBS snapshots)',
        'You configure (SG, SSL, auth)',
        'Manual (vertical or clustering)',
        'You set up (Prometheus, etc.)',
        'Manual integration',
        'EC2 + EBS costs',
        'Full control, existing Neo4j expertise'
    ],
    'Neo4j Aura': [
        'Managed Service',
        'Cypher',
        'Managed by Neo4j',
        '~5 minutes',
        'Automatic',
        'Managed SSL, auth',
        'Auto-scaling',
        'Built-in dashboard',
        'Via connection string',
        'Subscription tiers',
        'Cypher-first apps, quick setup'
    ]
}

df = pd.DataFrame(comparison_data)
df = df.set_index('Feature')
print("Graph Database Hosting Comparison")
print("=" * 80)
for idx, row in df.iterrows():
    print(f"\n{idx}:")
    for col in df.columns:
        print(f"  {col:20s} → {row[col]}")

## Step 7: Common Hosting Challenges — Graph DBs vs RDBMS vs NoSQL

The hosting challenges for graph databases are fundamentally the same as for any database system. Let's compare:

In [ ]:
challenges = {
    'Challenge': [
        'Network Isolation',
        'Authentication',
        'Encryption at Rest',
        'Encryption in Transit',
        'Automated Backups',
        'High Availability',
        'Scaling Reads',
        'Scaling Writes',
        'Monitoring',
        'Patch Management'
    ],
    'RDBMS (RDS/PostgreSQL)': [
        'VPC + Subnet Groups',
        'Username/Password, IAM',
        'KMS encryption',
        'SSL/TLS',
        'RDS automated snapshots',
        'Multi-AZ deployment',
        'Read replicas',
        'Vertical scaling',
        'CloudWatch + PI',
        'Maintenance windows'
    ],
    'NoSQL (DynamoDB/DocumentDB)': [
        'VPC endpoints / VPC',
        'IAM policies / passwords',
        'Default encrypted / KMS',
        'TLS required / optional',
        'Point-in-time / snapshots',
        'Multi-region / replica sets',
        'Auto-scaling / replicas',
        'Partition keys / sharding',
        'CloudWatch',
        'Serverless / windows'
    ],
    'Graph (Neptune/Neo4j)': [
        'VPC + Subnet Groups',
        'IAM / Neo4j auth',
        'KMS encryption',
        'SSL/TLS',
        'Continuous to S3 / manual',
        'Read replicas + failover',
        'Up to 15 read replicas',
        'Vertical scaling',
        'CloudWatch / custom',
        'Automatic / manual'
    ]
}

df = pd.DataFrame(challenges)
df = df.set_index('Challenge')
print("Database Hosting Challenges Comparison")
print("=" * 90)
for idx, row in df.iterrows():
    print(f"\n{idx}:")
    for col in df.columns:
        print(f"  {col:35s} → {row[col]}")

print("\n" + "=" * 90)
print("💡 KEY INSIGHT: The operational challenges are nearly identical across all database types.")
print("   The choice between managed vs self-hosted matters more than the database model!")

## Step 8: Clean Up Resources

⚠️ **IMPORTANT:** Run this section to delete all AWS resources and avoid unwanted charges!

Resources will be deleted in reverse order:
1. Neptune instance
2. Neptune cluster
3. DB subnet group
4. Security group

In [ ]:
# Delete Neptune instance
try:
    neptune_client.delete_db_instance(
        DBInstanceIdentifier=NEPTUNE_INSTANCE_ID,
        SkipFinalSnapshot=True
    )
    print(f"✅ Deleting Neptune instance: {NEPTUNE_INSTANCE_ID}")
except ClientError as e:
    if 'DBInstanceNotFound' in str(e):
        print(f"⚠️ Instance already deleted: {NEPTUNE_INSTANCE_ID}")
    else:
        print(f"❌ Error: {e}")

In [ ]:
# Wait for instance to be deleted before deleting the cluster
def wait_for_neptune_deleted(client, instance_id, max_wait=900):
    """Wait for Neptune instance to be fully deleted."""
    start_time = time.time()
    while time.time() - start_time < max_wait:
        try:
            response = client.describe_db_instances(
                DBInstanceIdentifier=instance_id
            )
            status = response['DBInstances'][0]['DBInstanceStatus']
            elapsed = int(time.time() - start_time)
            print(f"   ⏳ Status: {status} ({elapsed}s elapsed)")
        except ClientError as e:
            if 'DBInstanceNotFound' in str(e):
                print(f"   ✅ Instance deleted successfully!")
                return True
            raise
        time.sleep(30)
    return False

print("⏳ Waiting for Neptune instance to be deleted...")
wait_for_neptune_deleted(neptune_client, NEPTUNE_INSTANCE_ID)

In [ ]:
# Delete Neptune cluster
try:
    neptune_client.delete_db_cluster(
        DBClusterIdentifier=NEPTUNE_CLUSTER_ID,
        SkipFinalSnapshot=True
    )
    print(f"✅ Deleting Neptune cluster: {NEPTUNE_CLUSTER_ID}")
except ClientError as e:
    if 'DBClusterNotFoundFault' in str(e):
        print(f"⚠️ Cluster already deleted: {NEPTUNE_CLUSTER_ID}")
    else:
        print(f"❌ Error: {e}")

print("\n⏳ Waiting 60 seconds for cluster deletion to propagate...")
time.sleep(60)
print("✅ Done waiting.")

### Clean Up Network Resources

In [ ]:
# Delete DB subnet group
try:
    neptune_client.delete_db_subnet_group(
        DBSubnetGroupName=SUBNET_GROUP_NAME
    )
    print(f"✅ Deleted subnet group: {SUBNET_GROUP_NAME}")
except ClientError as e:
    if 'DBSubnetGroupNotFoundFault' in str(e):
        print(f"⚠️ Subnet group already deleted")
    else:
        print(f"❌ Error: {e}")

# Delete security group
try:
    ec2_client.delete_security_group(GroupId=sg_id)
    print(f"✅ Deleted security group: {sg_id}")
except ClientError as e:
    if 'InvalidGroup.NotFound' in str(e):
        print(f"⚠️ Security group already deleted")
    elif 'DependencyViolation' in str(e):
        print(f"⚠️ Security group still in use — Neptune may still be deleting.")
        print(f"   Wait a few minutes and re-run this cell.")
    else:
        print(f"❌ Error: {e}")

print("\n🎉 Cleanup complete!")

## Summary

### What You Learned

1. **AWS Neptune** — Amazon's fully managed graph database with built-in security, continuous backups, and support for Gremlin, SPARQL, and openCypher query languages
2. **Neo4j on AWS** — How to discover and deploy Neo4j via the AWS Marketplace, including AMIs and managed options (Aura)
3. **Infrastructure Patterns** — VPCs, security groups, and subnet groups are the same regardless of database type (RDBMS, NoSQL, or Graph)
4. **Query Language Portability** — Neptune's openCypher support means your Cypher skills from Exercises 1 & 2 transfer directly
5. **Hosting Trade-offs** — Managed vs self-hosted graph databases have the same trade-offs as managed vs self-hosted relational and NoSQL databases

### Key Takeaways

- Graph databases on AWS follow the **same operational patterns** as any other managed database
- **Neptune** is ideal when you want zero-ops management and native AWS integration
- **Neo4j on AWS** gives you full Cypher support and the rich Neo4j ecosystem
- The hosting challenges (security, backups, scaling, monitoring) are **universal** — they don't change based on data model
- Your choice should be driven by: query language preference, AWS integration needs, and operational capacity

### Next Steps

- Explore Neptune's **Neptune ML** feature for machine learning on graphs
- Try **Neo4j Aura** for a managed Cypher-first experience
- Apply graph modeling patterns from Exercise 2 to your Neptune or Neo4j deployment
- Consider **Neptune Serverless** for variable workloads with automatic scaling

## Bonus Reference: AWS CLI Equivalents

Everything we did with boto3 can also be done via the AWS CLI:

```bash
# Create Neptune cluster
aws neptune create-db-cluster \
    --db-cluster-identifier graph-exercise-cluster \
    --engine neptune \
    --db-subnet-group-name neptune-exercise-subnet-group \
    --vpc-security-group-ids sg-xxxxx

# Create Neptune instance
aws neptune create-db-instance \
    --db-instance-identifier graph-exercise-instance-1 \
    --db-cluster-identifier graph-exercise-cluster \
    --db-instance-class db.t3.medium \
    --engine neptune

# Check cluster status
aws neptune describe-db-clusters \
    --db-cluster-identifier graph-exercise-cluster

# Delete instance
aws neptune delete-db-instance \
    --db-instance-identifier graph-exercise-instance-1 \
    --skip-final-snapshot

# Delete cluster
aws neptune delete-db-cluster \
    --db-cluster-identifier graph-exercise-cluster \
    --skip-final-snapshot
```